In [0]:
%sql
describe table extended workspace.amazon_fullfilment_bronze.addresses


In [0]:
df_addresses = spark.table("workspace.amazon_fullfilment_bronze.addresses")
display(df_addresses.summary())

In [0]:
df_bin = spark.table("workspace.amazon_fullfilment_bronze.bin")
display(df_bin.summary())

In [0]:
df_customers = spark.table("workspace.amazon_fullfilment_bronze.customers")
display(df_customers.summary())

In [0]:
df_inventory = spark.table("workspace.amazon_fullfilment_bronze.inventory")
display(df_inventory.summary())

In [0]:
df_order_items = spark.table("workspace.amazon_fullfilment_bronze.order_items")
display(df_order_items.summary())


In [0]:
df_orders = spark.table("workspace.amazon_fullfilment_bronze.orders")
display(df_orders.summary())
df_products = spark.table("workspace.amazon_fullfilment_bronze.products")
display(df_products.summary())
df_shipment = spark.table("workspace.amazon_fullfilment_bronze.shipment")
display(df_shipment.summary())
df_shelves = spark.table("workspace.amazon_fullfilment_bronze.shelves")
display(df_shelves.summary())

In [0]:
df_robots = spark.table("workspace.amazon_fullfilment_bronze.robots")
display(df_robots.summary())

In [0]:
%sql
analyze table workspace.amazon_fullfilment_bronze.addresses compute statistics for all columns

Databricks data profile. Run in Databricks to view.

In [0]:
%sql
analyze table workspace.amazon_fullfilment_bronze.addresses compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.bin compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.customers compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.inventory compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.order_items compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.orders compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.products compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.robots compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.shelves compute statistics for all columns;
analyze table workspace.amazon_fullfilment_bronze.shipment compute statistics for all columns;


In [0]:
spark.read.table("workspace.amazon_fullfilment_bronze.addresses").count()

In [0]:
# data for address_silver table
from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable 

# 1. Read from the Bronze Streaming Table
bronze_address_stream = spark.readStream \
    .table("workspace.amazon_fullfilment_bronze.addresses")

def upsert_to_silver(batch_df, batch_id):
    # 2. Filter for Quality: Ensure City and Province are NOT NULL
    # Records failing this will be skipped (dropped)
    clean_batch_df = batch_df.filter(
        col("city").isNotNull() & col("province").isNotNull()
    ).withColumn("Updated_Timestamp", current_timestamp())

    silver_table = DeltaTable.forName(spark, "workspace.amazon_fullfilment_silver.addresses_silver")
    # 3. Perform the SCD Type 1 Merge (Overwrite on ID match)
    (silver_table.alias("target")
        .merge(
            clean_batch_df.alias("source"),
            "source.customer_ID = target.customer_ID AND source.street = target.street AND source.province = target.province AND source.postal_code = target.postal_code"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


# 4. Start the Stream
query = (bronze_address_stream.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", "/Volumes/workspace/amazon_fullfilment_silver/processing_zone/address_silver")
    .trigger(availableNow=True)
    .start())

In [0]:
# data for address_silver table
from pyspark.sql.functions import col, current_timestamp

def upsert_to_silver(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    # 1. Capture the correct session for this specific batch
    # This is the "Spark Connect Safe" way
    batch_spark = batch_df.sparkSession 

    # 2. Clean and Filter
    clean_batch_df = batch_df.filter(
        col("city").isNotNull() & col("province").isNotNull()
    ).withColumn("Updated_Timestamp", current_timestamp())

    # 3. Register view in the BATCH session, not the global one
    clean_batch_df.createOrReplaceTempView("batch_updates")

    # 4. Use batch_spark instead of spark
    batch_spark.sql("""
        MERGE INTO workspace.amazon_fullfilment_silver.address_silver AS target
        USING batch_updates AS source
        ON source.customer_ID = target.customer_ID 
        WHEN MATCHED THEN
          UPDATE SET target.Address_ID=source.Address_ID
          , target.Customer_ID=source.Customer_ID
          , target.Street_address=source.Street
          , target.City=source.City
          , target.Province=source.Province
          , target.Postal_Code=source.Postal_Code
          , target.Updated_Timestamp=source.Updated_Timestamp
        WHEN NOT MATCHED THEN
          INSERT (Address_ID, Customer_ID, Street_address, City, Province, Postal_Code, Updated_Timestamp)
          VALUES (source.Address_ID, source.Customer_ID, source.Street, source.City, source.Province, source.Postal_Code, source.Updated_Timestamp)
    """)
    print(f"Batch {batch_id} completed.")

# 5. Start the Stream
checkpoint_path = "/Volumes/workspace/amazon_fullfilment_silver/processing_zone/address_silver"
dbutils.fs.rm(checkpoint_path, True) # Clear stale metadata

query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.addresses")
    .writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.address_silver
